# Predict Representations, Not Every Pixel

Separate JEPA-family mechanisms, collapse controls, and multimodal uncertainty.

## Before You Start

Course 13. Entry course: 12. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** embodied-03_actions.

## What You Will Be Able to Do

- Compute latent prediction loss
- Update an EMA target correctly
- Distinguish EMA designs from end-to-end regularized variants

## Why the average future can be blurry

When the same context permits two different future images, a deterministic pixel-MSE predictor minimizes expected loss by predicting their conditional mean. If an object can move left or right, the mean can contain a faint object in both places rather than one plausible future. A generative distribution can represent multiple futures; a latent predictor can instead focus on features that are useful and predictable. Neither choice removes uncertainty automatically.

A joint-embedding predictive architecture encodes context and targets, then predicts target representations from context, sometimes conditioned on an action or mask. This differs from a joint-embedding similarity objective and from reconstructing every target pixel. The representation must retain what the downstream task needs; discarding unpredictable detail can also discard something important for control.

## The target branch is a design choice

One common design uses a stop-gradient target encoder updated by exponential moving average: target <- momentum*target+(1-momentum)*online. Gradients train the online encoder and predictor, not the target through that loss. The browser lab computes masked latent MSE and a nonmutating EMA update. This teaches that design, not a universal definition of JEPA.

LeWorldModel uses a different end-to-end recipe: next-embedding prediction plus SIGReg, which encourages Gaussian-like latent distributions through random projections. It does not require EMA or stop-gradient. Our toy variance regularizer is not SIGReg. The comparison notebook labels these differences explicitly and treats collapse diagnostics, prediction error, and downstream planning success as distinct tests.

For masked latent MSE, convert predictions and targets to floating point before subtraction and squaring. Select Boolean target positions, then average over both selected positions and feature dimensions. An int16 value of 200 has squared error 40000 against zero; unsigned subtraction must also preserve negative differences before squaring. An empty selection is an error, and neither targets nor the mask should be modified.

## Worked Example: Update a slowly moving target

Old target weight=2, online weight=6, momentum=.9.

1. **Retain history:** .9*2=1.8.
2. **Add online contribution:** .1*6=.6.
3. **New target:** 1.8+.6=2.4. The target did not take a direct optimizer step through this loss.

Write down the gradient paths and update rules; similar architecture diagrams can train differently.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Masked latent prediction and EMA target mechanics."""
import numpy as np

def latent_mse(prediction, target, mask):
    """Mean squared feature error over selected [B,T] target positions.

prediction/target matching finite [B,T,D], Boolean mask [B,T]. Reject
mismatched shapes, nonfinite values, or empty mask with ValueError.
Average over selected positions AND D features. Do not mutate arrays.
Convert prediction and target to floating point before arithmetic."""
    raise NotImplementedError('Implement this function, then run the checks.')

def ema_target(target, online, momentum):
    """Return fresh m*target+(1-m)*online. Matching finite arrays assumed.

Require finite 0<=momentum<=1, else ValueError. No input mutation.
This is an update rule, not proof of noncollapse."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Two equally likely scalar futures are -2 and +2 under squared error.

- Optimal deterministic prediction

### Worked Answers

- 0: The conditional mean is 0, even though neither actual future equals 0.

## Experiment

Run `experiments/embodied_world.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Compare an ambiguous pixel mean with a distribution over distinct futures.
- The toy world experiment uses fixed observations and learned dynamics; it does not claim to reproduce end-to-end LeWorldModel representation learning.

## Transfer Problems

1. Draw which arrays receive gradients in an EMA-target design and in a jointly trained regularized design.
2. Give a task where averaging away an unpredictable feature would be harmful.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Revisiting Feature Prediction from Video](https://arxiv.org/abs/2404.08471). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.

- [LeWorldModel](https://arxiv.org/abs/2603.19312)


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Masked latent prediction and EMA target mechanics."""
import numpy as np

def latent_mse(prediction, target, mask):
    """Mean squared feature error over selected [B,T] target positions.

prediction/target matching finite [B,T,D], Boolean mask [B,T]. Reject
mismatched shapes, nonfinite values, or empty mask with ValueError.
Average over selected positions AND D features. Do not mutate arrays.
Convert prediction and target to floating point before arithmetic."""
    raise NotImplementedError('Implement this function, then run the checks.')

def ema_target(target, online, momentum):
    """Return fresh m*target+(1-m)*online. Matching finite arrays assumed.

Require finite 0<=momentum<=1, else ValueError. No input mutation.
This is an update rule, not proof of noncollapse."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
p = np.array([[[1.,3.],[100.,100.]],[[2.,4.],[5.,7.]]])
t = np.zeros_like(p);m = np.array([[1,0],[0,1]],bool)
assert latent_mse(p,t,m) == (1+9+25+49)/4
for dtype in (np.int16,np.uint8):
    assert latent_mse(np.array([[[200]]],dtype=dtype),np.array([[[0]]],dtype=dtype),np.array([[True]])) == 40000.
    prediction = np.array([[[200,0],[100,100]]],dtype=dtype)
    target = np.array([[[0,200],[0,0]]],dtype=dtype)
    selected = np.array([[True,False]])
    originals = [value.copy() for value in (prediction,target,selected)]
    assert latent_mse(prediction,target,selected) == 40000., 'Integer differences and squares must use floating arithmetic'
    assert all(np.array_equal(value,before) for value,before in zip((prediction,target,selected),originals))
a,b = np.array([2.,4.]),np.array([6.,0.])
result = ema_target(a,b,.9)
assert np.allclose(result,[2.4,3.6])
result[:] = 0
assert np.array_equal(a,[2,4]) and np.array_equal(b,[6,0])
for prediction,target,selected in [(p,t,np.zeros_like(m)),(p,t[:1],m),
                                    (p,t,m.astype(int)),(p+np.inf,t,m)]:
    try:
        latent_mse(prediction,target,selected)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid latent targets, shapes, and masks')
print('Latent prediction checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Masked latent prediction and EMA target mechanics."""
import numpy as np


def latent_mse(prediction, target, mask):
    """Mean squared feature error over selected [B,T] target positions.

    prediction/target matching finite [B,T,D], Boolean mask [B,T]. Reject
    mismatched shapes, nonfinite values, or empty mask with ValueError.
    Average over selected positions AND D features. Do not mutate arrays.
    Convert prediction and target to floating point before arithmetic.
    """
    p,t,m = np.asarray(prediction,float),np.asarray(target,float),np.asarray(mask)
    if p.ndim != 3 or p.shape != t.shape or m.shape != p.shape[:2] or m.dtype.kind != 'b' or not m.any() or p.shape[-1] == 0 or not np.isfinite(p).all() or not np.isfinite(t).all():
        raise ValueError('invalid prediction target or mask')
    return float(((p[m]-t[m])**2).mean())


def ema_target(target, online, momentum):
    """Return fresh m*target+(1-m)*online. Matching finite arrays assumed.

    Require finite 0<=momentum<=1, else ValueError. No input mutation.
    This is an update rule, not proof of noncollapse.
    """
    if not np.isfinite(momentum) or not 0 <= momentum <= 1:
        raise ValueError('invalid momentum')
    return momentum*target+(1-momentum)*online


In [ ]:
import numpy as np
p = np.array([[[1.,3.],[100.,100.]],[[2.,4.],[5.,7.]]])
t = np.zeros_like(p);m = np.array([[1,0],[0,1]],bool)
assert latent_mse(p,t,m) == (1+9+25+49)/4
for dtype in (np.int16,np.uint8):
    assert latent_mse(np.array([[[200]]],dtype=dtype),np.array([[[0]]],dtype=dtype),np.array([[True]])) == 40000.
    prediction = np.array([[[200,0],[100,100]]],dtype=dtype)
    target = np.array([[[0,200],[0,0]]],dtype=dtype)
    selected = np.array([[True,False]])
    originals = [value.copy() for value in (prediction,target,selected)]
    assert latent_mse(prediction,target,selected) == 40000., 'Integer differences and squares must use floating arithmetic'
    assert all(np.array_equal(value,before) for value,before in zip((prediction,target,selected),originals))
a,b = np.array([2.,4.]),np.array([6.,0.])
result = ema_target(a,b,.9)
assert np.allclose(result,[2.4,3.6])
result[:] = 0
assert np.array_equal(a,[2,4]) and np.array_equal(b,[6,0])
for prediction,target,selected in [(p,t,np.zeros_like(m)),(p,t[:1],m),
                                    (p,t,m.astype(int)),(p+np.inf,t,m)]:
    try:
        latent_mse(prediction,target,selected)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid latent targets, shapes, and masks')
print('Latent prediction checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Learn action-conditioned dynamics, plan, and compare predictions to a simulator.

Known linear dynamics are the trusted simulator. Latents are observed states;
this is NOT an end-to-end pixel encoder or LeWorldModel/SIGReg reproduction.
"""
import json
import numpy as np


def world_experiment():
    rng=np.random.default_rng(18)
    true_a=np.array([[.9,.1],[0.,.95]]);true_b=np.array([[.5,0.],[.1,.4]])
    states=rng.normal(size=(300,2));actions=rng.uniform(-1,1,size=(300,2))
    next_states=states@true_a.T+actions@true_b.T
    noisy=next_states+rng.normal(size=next_states.shape)*.03
    design=np.column_stack([states,actions])
    fitted=np.linalg.lstsq(design[:240],noisy[:240],rcond=None)[0]
    learned_a,learned_b=fitted[:2].T,fitted[2:].T
    heldout=float(((design[240:]@fitted-next_states[240:])**2).mean())
    initial=np.array([-1.,-.5]);goal=np.array([1.,1.])
    candidates=rng.uniform(-1,1,size=(512,8,2))
    def costs(a,b,candidates,start):
        z=np.repeat(start[None,:],len(candidates),0);cost=np.zeros(len(candidates))
        for t in range(candidates.shape[1]):
            z=z@a.T+candidates[:,t]@b.T
            cost+=((z-goal)**2).sum(-1)+.02*(candidates[:,t]**2).sum(-1)
        return cost
    predicted=costs(learned_a,learned_b,candidates,initial)
    observed=costs(true_a,true_b,candidates,initial);choice=int(predicted.argmin())
    error_by_horizon=[]
    z_true,z_model=initial.copy(),initial.copy()
    for action in candidates[choice]:
        z_true=true_a@z_true+true_b@action;z_model=learned_a@z_model+learned_b@action
        error_by_horizon.append(float(((z_true-z_model)**2).mean()))
    # The replanning trace has a different budget; see the returned limitation.
    bad_b=learned_b*1.8
    state=initial.copy();closed=[]
    for _ in range(8):
        local=rng.uniform(-1,1,size=(128,4,2))
        chosen=int(costs(learned_a,bad_b,local,state).argmin())
        state=true_a@state+true_b@local[chosen,0];closed.append(state.tolist())
    # Conditional pixel-MSE mean for two distinct futures.
    futures=np.array([[1.,0.,0.],[0.,0.,1.]])
    return dict(seed=18,train_transitions=240,test_transitions=60,heldout_one_step_mse=heldout,
        learned_a=learned_a.tolist(),learned_b=learned_b.tolist(),selected_plan=choice,
        predicted_cost=float(predicted[choice]),simulator_cost=float(observed[choice]),
        multistep_mse=error_by_horizon,replanned_states=closed,ambiguous_pixel_mean=futures.mean(0).tolist(),
        limitation='Observed linear states, bounded sampled actions. Replanning uses extra observations and planning work; not a matched-cost superiority claim.')


if __name__=='__main__':
    print(json.dumps(world_experiment(),indent=2))
